# Notebook 1: Create Synthetic ESP Sensor Data

This notebook generates realistic ESP (Electrical Submersible Pump) sensor readings for 40 horizontal wells in the Permian Basin. The synthetic data simulates real-world sensor behavior including pressure cycling, temperature fluctuations, and natural measurement noise.

**Pipeline Position:** This is the first step — it creates the raw sensor data that downstream notebooks use for feature engineering, model training, and inference.

**Output:** `ENERGY_DEMO.WELLS.WELL_SENSORS` — ~58,560 rows (40 wells × 1,464 readings at 6-hour intervals over ~1 year)

## 1. Connect to Snowflake

Establish a Snowpark session using shared connection utilities. The session targets the `ENERGY_DEMO.WELLS` schema where all well data resides.

In [ ]:
import sys

sys.path.insert(
    0, "/Users/trasmith/Documents/Trace/Code/cortex-search-energy-demo/notebooks"
)

from utils import get_session, DATABASE, SCHEMA

session = get_session()
print(f"Connected to {DATABASE}.{SCHEMA}")

## 2. Check Existing Well Metadata

Verify that the `WELL_METADATA` table is populated with our 40 horizontal wells (Wolfcamp, Spraberry, and Bone Spring formations). The `TVD_FT` (True Vertical Depth) column is used later to scale sensor readings — deeper wells produce higher pressures.

In [ ]:
df_wells = session.sql(
    "SELECT API_NO, WELL_NAME, TVD_FT FROM WELL_METADATA ORDER BY API_NO"
).to_pandas()
print(f"Wells found: {len(df_wells)}")
df_wells.head(10)

## 3. Generate Sensor Data

Create the `WELL_SENSORS` table entirely in Snowflake SQL for maximum performance. Each well gets 6-hour readings from Feb 2025 to Feb 2026. The generation logic simulates:

- **Base values** scaled by TVD (deeper wells = higher pressures and temperatures)
- **Sinusoidal patterns** mimicking pump cycling and day/night temperature effects
- **Random noise** for natural sensor variation

Sensor columns: `INTAKE_PRESSURE_PSI`, `DISCHARGE_PRESSURE_PSI`, `MOTOR_TEMP_F`, `MOTOR_AMPS`, `VIBRATION_IPS`, `WELLHEAD_PRESSURE_PSI`, `WELLHEAD_TEMP_F`, `FREQUENCY_HZ`

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE WELL_SENSORS AS
WITH well_list AS (
    SELECT API_NO, WELL_NAME, TVD_FT FROM WELL_METADATA
),
date_range AS (
    SELECT DATEADD('hour', SEQ4() * 6, '2025-02-01'::TIMESTAMP) AS READING_TS
    FROM TABLE(GENERATOR(ROWCOUNT => 1464))
),
base AS (
    SELECT
        w.API_NO, w.WELL_NAME, d.READING_TS, w.TVD_FT,
        ROW_NUMBER() OVER (PARTITION BY w.API_NO ORDER BY d.READING_TS) AS rn,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) AS r1,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) AS r2,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) AS r3,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) AS r4,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) AS r5,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) AS r6,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) AS r7,
        UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) AS r8
    FROM well_list w CROSS JOIN date_range d
)
SELECT
    API_NO, WELL_NAME, READING_TS,
    ROUND(1800 + (TVD_FT/10.0)*r1 + 200*SIN(rn/50.0) + (r2-0.5)*300, 1) AS INTAKE_PRESSURE_PSI,
    ROUND(2800 + (TVD_FT/15.0)*r3 + 150*SIN(rn/60.0) + (r4-0.5)*200, 1) AS DISCHARGE_PRESSURE_PSI,
    ROUND(220 + 40*r5 + 15*SIN(rn/80.0) + (r6-0.5)*20, 1) AS MOTOR_TEMP_F,
    ROUND(45 + 20*r1 + 5*SIN(rn/40.0) + (r7-0.5)*10, 1) AS MOTOR_AMPS,
    ROUND(0.15 + 0.2*r2 + 0.05*ABS(SIN(rn/100.0)) + (r8-0.5)*0.1, 3) AS VIBRATION_IPS,
    ROUND(350 + 200*r3 + 50*SIN(rn/70.0) + (r4-0.5)*80, 1) AS WELLHEAD_PRESSURE_PSI,
    ROUND(135 + 25*r5 + 8*SIN(rn/90.0) + (r6-0.5)*10, 1) AS WELLHEAD_TEMP_F,
    ROUND(45 + 10*r7 + 3*SIN(rn/120.0) + (r8-0.5)*5, 1) AS FREQUENCY_HZ
FROM base
""").collect()
print("WELL_SENSORS created!")

## 4. Verify

Confirm the table was created with the expected row count, number of distinct wells, and date range coverage.

In [ ]:
count = session.sql("SELECT COUNT(*) AS CNT FROM WELL_SENSORS").to_pandas()["CNT"][0]
stats = session.sql("""
    SELECT COUNT(DISTINCT API_NO) AS WELLS, MIN(READING_TS) AS FIRST_TS, MAX(READING_TS) AS LAST_TS
    FROM WELL_SENSORS
""").to_pandas()

print(f"Total records: {count:,}")
print(f"Wells: {stats['WELLS'][0]}")
print(f"Date range: {stats['FIRST_TS'][0]} to {stats['LAST_TS'][0]}")

## 5. Preview Sample Data

Spot-check a few rows from a single well to confirm the generated values look reasonable.

In [ ]:
session.sql("""
    SELECT * FROM WELL_SENSORS
    WHERE WELL_NAME = 'Snowflake 1H'
    ORDER BY READING_TS DESC LIMIT 5
""").to_pandas()